Demo for conditionla workflow

# Non LLM

start -> formal post<br>->analyze content <br>->approve post, flag for review , reject post<br> end


In [ ]:
from typing import TypedDict, Literal

scam_words = ["buy now", "buynow", "free money", "click here"]

class PostDetails(TypedDict):
    userType : str
    post : str
    postStatus : Literal["approve","reject","review"]
    result : str

def readPost(state:PostDetails):
    post = input("Enter post:")
    return{
        "post" : post
    }

def analyzePost(state:PostDetails):
    content = state["post"]
    
    if any(word in content for word in scam_words) :
        postStatus = "reject"
    elif state["userType"] == "new user":
        postStatus = "review"
    else:
        postStatus = "approve"
    return{
        "postStatus" : postStatus
    }



In [ ]:
def approvePost(state:PostDetails):
    result = 'Post published successfully to the timeline.'
    return{
        "result" : result
    }
def rejectPost(state:PostDetails):
    result = 'Post is rejected since its scam.'
    return{
        "result" : result
    }
def reviewPost(state:PostDetails):
    result = 'Post is flagged for review since user is new.'
    return{
        "result" : result
    }

In [ ]:
def routingCondition(state:PostDetails)-> Literal["approvePost","rejectPost","reviewPost"]:
    if state["postStatus"] == "approve":
        return "approvePost"
    elif state["postStatus"] == "Reject":
        return "rejectPost"
    else:
        return "reviewPost"

In [ ]:
from langgraph.graph import StateGraph, START,END

graph = StateGraph(state_schema=PostDetails)

#add nodes
graph.add_node("readPost",readPost)
graph.add_node("analyzePost",analyzePost)
graph.add_node("approvePost",approvePost)
graph.add_node("rejectPost",rejectPost)
graph.add_node("reviewPost",reviewPost)

#add edges
graph.add_edge(START, "readPost")
graph.add_edge("readPost", "analyzePost")
graph.add_conditional_edges("analyzePost",routingCondition,{"approvePost":"approvePost","rejectPost":"rejectPost","reviewPost":"reviewPost"})

graph.add_edge("approvePost",END)
graph.add_edge("rejectPost",END)
graph.add_edge("reviewPost",END)

workout = graph.compile()

final_state = workout.invoke({"userType":"existing"})
for key, val in final_state.items():
    print(f"{key}:{val}")


In [ ]:
from IPython.display import Image

Image(workout.get_graph().draw_mermaid_png())

# LLM 

In [ ]:
import os 
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-20b")



In [ ]:
from pydantic import BaseModel, Field
from typing import Literal, TypedDict
 
class SentimentSchema(BaseModel):
    sentiment : Literal["positive","negative"] = Field(description="Sentiment of the review")

class DiagnosisSchema(BaseModel):
    issue : Literal["UI","Product","Bug","Support","Other"] = Field(description="The category of issue mentioned in the review")
    tone : Literal["angry","disappointment","frustrated","calm"] = Field(description="The emotional tone expressed by user")
    urgency : Literal["low","medium","high"] = Field(description="How urgent or critcal the issue appears to be")

In [ ]:
structured_model1 = llm.with_structured_output(SentimentSchema)
structured_model2 = llm.with_structured_output(DiagnosisSchema)

In [ ]:
class ReviewState(TypedDict):
    review : str
    sentiment : Literal["positive","negative"]
    dignosis : DiagnosisSchema
    response : str

def find_sentiment(state : ReviewState):
    prompt = f"Find the sentiment for the review : {state['review']}"
    return{
        "sentiment" : structured_model1.invoke(prompt).sentiment
    }

def positive_resposne(state : ReviewState):
    prompt = f"Please write thank you message for this positve revoew {state['review']}"
    return{
        "response" : llm.invoke(prompt).content
    }

def run_diagnosis(state : ReviewState):
    prompt = f"Diagnose the negative review {state['review']}  and return issue, tone and urgency."
    response = structured_model2.invoke(prompt)
    return{
        "dignosis" : response.model_dump()
    }

def negative_resposne(state : ReviewState):
    dignosis = state["dignosis"]
    prompt = f"You are support assistant. The user has {dignosis['issue']},sounded {dignosis['tone']} and marked urgency as {dignosis['urgency']} for review : {state['review']}.Write an emphathetic and helpful resolution message."
    return{
        "response" : llm.invoke(prompt).content
    }

In [ ]:
def routing_condition(state : ReviewState)-> Literal["positive_resposne","run_diagnosis"]:
    sentiment = state["sentiment"]
    if sentiment == "positive":
        return "positive_resposne"
    else :
        return "run_diagnosis"

In [ ]:
from langgraph.graph import StateGraph, START, END

graph = StateGraph(state_schema=ReviewState)

# add nodes
graph.add_node("find_sentiment",find_sentiment)
graph.add_node("positive_resposne",positive_resposne)
graph.add_node("run_diagnosis",run_diagnosis)
graph.add_node("negative_resposne",negative_resposne)

# add edges
graph.add_edge(START,"find_sentiment")
graph.add_conditional_edges("find_sentiment",routing_condition,{"positive_resposne":"positive_resposne","run_diagnosis":"run_diagnosis"})
graph.add_edge("run_diagnosis","negative_resposne")
graph.add_edge("negative_resposne",END)
graph.add_edge("positive_resposne",END)

workout = graph.compile()

reviews= [
    {"review":"I absolute love this security camera. The 1480p HD video quality is crystal clear both day and night, and the motion detection is incredibly precise without triggering false alerts. Setting it up took less than five minutes using the mobile app. The battery life has been phenomenal, lasting over three months on a single charge. It integrates perfectly with my existing smart home system, and the two-way audio is loud and clear. I highly recommend this product to anyone looking for a reliable, high-quality home security solution!"},
    {"review":"DEFECTIVE DEVICE AND ZERO SUPPORT! FIX THIS IMMEDIATELY!!I am absolutely furious! This camera completely stopped working last night right after a forced firmware update, and now my entire driveway is unmonitored! The status light is blinking red, it won't reset, and the app completely crashes every time I try to reconnect it. This is a massive safety risk for my family! I have called your customer service hotline five times and have been stuck on hold for over an hour, and nobody is answering my urgent support tickets! I need a senior technician or support manager to contact me via email or phone RIGHT NOW to issue a replacement or a full refund! Do not buy this product if you value your security or your sanity!"}
]

for rev in reviews:
    print(workout.invoke(rev)["response"])

In [ ]:
from IPython.display import Image

Image(workout.get_graph().draw_mermaid_png())